# Triton is not about math first — it’s about mapping work to hardware
# 1️⃣ Python View
# Python code
───────────

    for i in range(0, N, B):
        y[i : i+B] = f(x[i : i+B])

# Mental picture

    x:  [ 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | ... ]
      └─────── slice ───────┘
               size = B

    Each loop iteration:
    grabs a slice
    processes it
    moves on

# 2️⃣ GPU View
    GPU launch
    ──────────
    ┌────────┐ ┌────────┐ ┌────────┐ ┌────────┐
    │ Block 0│ │ Block 1│ │ Block 2│ │ Block 3│ ...
    └────────┘ └────────┘ └────────┘ └────────┘

    Each block:
    works independently
    has its own chunk of data
    runs the same program

# 3️⃣ Triton View
    A Triton kernel runs once per block, not per element

    program_id = 0      program_id = 1      program_id = 2
    ┌────────────┐      ┌────────────┐      ┌────────────┐
    │ Triton     │      │ Triton     │      │ Triton     │
    │ program    │      │ program    │      │ program    │
    └────────────┘      └────────────┘      └────────────┘

    Inside a single Triton program:

    program_id = pid
    BLOCK_SIZE = B


    You recreate slicing manually:

    offsets = pid * B + [0, 1, 2, ..., B-1]

    Visual
    Global memory (x):

    [ 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | ... ]
            ↑
            pid * B

    Offsets inside block:
            [0 | 1 | 2 | 3]

    Final addresses:
            [4 | 5 | 6 | 7]

# Side-by-Side Mapping

    Python                          Triton / GPU
    ─────────────────────────────────────────────────
    for i in range(...)        →    grid of programs
    i                          →    program_id
    x[i:i+B]                   →    pid * B + offsets
    single function call       →    thousands of blocks
    loop iteration             →    one Triton program


# Mental Model
    
    Python / PyTorch	GPU / Triton
    for-loop	grid of programs
    slice [i:i+B]	block of threads
    Python index	program_id + offsets
    one function call	thousands of parallel programs


Triton does not “create blocks” during a forward pass.
The GPU runtime launches them all at once

## Moving from Pytorch to kernel code

In [14]:
N = 1024
block_size = 8
Z= [0] * N
X = [0] * N
Y = [0] * N

# Stage 1 — Solidify Block Thinking (Pure PyTorch)

In [7]:
# Manual indexing (no slicing)

for i in range(0,N,block_size):
    for j in range(block_size):  #this loop will run for each i
        id = i+j
        if id < N:
            Z[id] = X[id] + Y[id]

In [9]:
# Multiple “programs”
# will write this in pytorch/python assume it is a gpu level code
# This is exactly how Triton thinks.
number_of_programs = (N + block_size-1) // block_size

for pid in range(number_of_programs):
    start = pid * block_size
    end = min(start + block_size, N)
    Z[start:end] = X[start:end] + Y[start:end]

# Stage 2 — Block Offsets

some docs for understanding this -

## Triton cannot use Python slicing:
    X[start:end]   ❌ not allowed
    Instead, Triton works like this:
        Every program must explicitly list the memory addresses it will touch

Offsets = “manual slice”

    Let’s rewrite one block of your loop without slicing.

        Your slice version (block-local view)
        start = pid * block_size
        end = start + block_size
        indices = start, start+1, start+2, ...

    Explicit version (this is offsets)
        offsets = pid * block_size + [0, 1, 2, ..., block_size-1]

Example - 
        pid = 2
        block_size = 4

        offsets = 2*4 + [0,1,2,3]
                = [8, 9, 10, 11]

        1. Offsets are just explicit indices
        2. They replace slicing

    Why offsets must be a vector (not a scalar)

    In Triton:

        a program handles many elements at once
        each element has its own address
        So instead of:

        i = start
        You have:
        i = [start, start+1, start+2, ...]

    That vector is called offsets.

## Now the mask-

    Let’s look at the last block.
    Example - 
        N = 10
            block_size = 4
            number_of_programs = 3
        Blocks:
            pid 0 → [0,1,2,3]
            pid 1 → [4,5,6,7]
            pid 2 → [8,9,10,11]  ← PROBLEM

    Indices 10 and 11 are out of bounds.
    In Python you fixed this with:
    end = min(start + block_size, N)
    -  In Triton, you cannot change the block size dynamically.
        offsets = [8, 9, 10, 11]
        mask    = [T, T,  F,  F]    - mask will tell wich positions to care about ## as a example you can see in decoder some tokens are also masked

In [ ]:
# Explicit offsets
# Instead of slices, build offsets:

import torch
pid = 0
offsets = pid * block_size + torch.arange(block_size)   #-> tensor([0, 1, 2, 3, 4, 5, 6, 7])
mask = offsets < N

Z[offsets[mask]] = X[offsets[mask]] + Y[offsets[mask]]

In [ ]:
# Loop over program IDs
# using this idea of offsets

number_of_programs = (N + block_size - 1) // block_size

for pid in range(number_of_programs):
    offsets = pid*block_size + torch.arange(block_size)
    mask = offsets < N
    Z[offsets[mask]] = X[offsets[mask]] + Y[offsets[mask]]  ## for triton based code where the container accept a tensor this is correct code, as it is in python and assuming list but the offsets is a tensor

TypeError: only integer tensors of a single element can be converted to an index

## Stage 3 — “Pretend Triton” in PyTorch

In [16]:
# Vector scaling, Triton-style
a = 3

for pid in range(number_of_programs):
    offsets = pid*block_size + torch.arange(block_size)
    mask = offsets < N
    Y[offsets[mask]] = a * X[offsets[mask]]

TypeError: only integer tensors of a single element can be converted to an index

In [19]:
# Fused add + scale
X = torch.arange(N)
Y = torch.arange(N)
Z = torch.zeros_like(X)

for pid in range(number_of_programs):
    offsets = pid * block_size + torch.arange(block_size)
    mask = offsets < N
    Z[offsets[mask]] = a * X[offsets[mask]] + Y[offsets[mask]]

# Stage 4 — First Triton

triton.cdiv(N, BLOCK) is equivalent to 

    number_of_programs = (N + BLOCK - 1) // BLOCK
    for pid in range(number_of_programs):

What grid actually is- 

    grid is not code.
    It is a declaration.

    You are telling Triton:

    “Launch this kernel this many times.”

pid = tl.program_id(0)
Reads a hardware-provided value.

In [ ]:
# Triton vector add

## writing code for vector addition
import triton
import triton.language as tl
device = "cuda" if torch.cuda.is_available() else "cpu"

@triton.jit
def add_two_tensor_kernel(X,Y,Z,N, block_size: tl.constexpr):
  pid = tl.program_id(0)
  offsets = pid * block_size + tl.arange(0,block_size)
  mask = offsets < N
  x = tl.load(X + offsets, mask = mask)
  y = tl.load(Y + offsets, mask = mask)
  tl.store(pointer = Z + offsets, value = x+y, mask = mask)


X = torch.arange(N, device = device)
Y = torch.arange(N, device = device)
Z = torch.empty_like(X)


grid = (triton.cdiv(N, block_size),)
add_two_tensor_kernel[grid](X,Y,Z,N, block_size = 8)

In [ ]:

## writing the row sum kernel
@triton.jit
def row_sum_kernel(X_ptr, ## pointer to the x input
                   Y_ptr, ## pointer to the Y output
                   D, ## number of columns
                   stride, ## stride between the rows
                   Block: tl.constexpr):
  
  pid = tl.program_id(0)
  offset = tl.arange(0,Block)
  mask = offset < D

  row_start_ptr = X_ptr + pid * stride
  ## load the row

  row = tl.load(row_start_ptr + offset, mask = mask)

  row_sum = tl.sum(row, axis = 0)

  tl.store(Y_ptr + pid, row_sum)


In [ ]:
#input
X = torch.rand(3,4,device=device)
B , D = X.shape
Block = 8

# output
Y = torch.empty(B, device=device)

grid = (B,)

row_sum_kernel[grid](X,Y,D,stride = x.stride(0),Block= Block)